<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Multi-Agent Creative Story Building System </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
This notebook demonstrates how to build a **creative writing multi-agent system** where specialized agents collaborate to create high-quality fictional stories.

The system uses:
- Specialized creative agents
- Role-specific prompts
- Shared story memory
- Agent communication
- Iterative story improvement

**Goal:** Generate a coherent, creative story from a simple user idea.

Agents collaborate to:
- Plan the story structure
- Design characters
- Build the world
- Write scenes
- Maintain consistency
- Improve quality

**Example Input:**  
`"A cyberpunk detective discovers memories can be stolen."`

**Example Output:** A polished multi-chapter story with consistent plot, characters, and world-building.

---

## 1. Setup

In [ ]:
# Cell 2 — Environment Setup
# Install required libraries and configure API keys

!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" --force-reinstall
!pip install -q langsmith

import os
import json
from getpass import getpass
from typing import Dict, List, Optional

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

## 2. Defining Agent Roles

We will create a **Creative Story Team** with six specialized roles:

1. **Story Planner Agent** — Understands the story idea, decides genre & tone, creates a plot outline (beginning, middle, climax, ending). Output: High-level story blueprint.

2. **Character Designer Agent** — Creates main characters with motivations, flaws, backstories, and relationships. Output: Character profiles.

3. **World Builder Agent** — Designs the story setting: rules, locations, culture, technology/magic systems. Output: World-building document.

4. **Scene Writer Agent** — Converts the outline into narrative prose with dialogue, pacing, and vivid descriptions. Output: Story chapters/scenes.

5. **Consistency Checker Agent** — Detects contradictions, plot holes, character inconsistencies, and timeline issues. Output: Corrections & revisions.

6. **Style Editor Agent** — Improves writing quality, enhances dialogue, strengthens emotional impact. Output: Polished final story.

In [ ]:
# Cell 4 — Imports + Tool Definitions

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client
import random

# ─────────────────────────────────────────────────────────────────────────────
# Shared Story Memory — a global store that agents use to save and retrieve
# characters, locations, story facts, and important events.
# ─────────────────────────────────────────────────────────────────────────────
story_memory: Dict[str, List[str]] = {
    "characters": [],
    "locations": [],
    "facts": [],
    "events": [],
}


# ─────── Tool 1: Genre Inspiration Tool ───────
@tool
def genre_inspiration(genre: str) -> str:
    """Provide storytelling inspiration based on genre. Returns themes, tropes, and ideas for: fantasy, sci-fi, mystery, horror, cyberpunk, romance, thriller."""
    inspirations = {
        "fantasy": {
            "themes": ["chosen one", "power corrupts", "nature vs. industry", "legacy of ancient magic"],
            "tropes": ["prophecy", "magic academy", "dark lord", "enchanted artifact", "quest"],
            "ideas": ["A spell that rewrites history every time it's cast", "A kingdom where dreams are currency"]
        },
        "sci-fi": {
            "themes": ["identity in technology", "ethics of AI", "colonization", "time paradox"],
            "tropes": ["generation ship", "rogue AI", "first contact", "cybernetic uprising"],
            "ideas": ["A planet where gravity shifts every 12 hours", "Humans who upload consciousness to escape a dying Earth"]
        },
        "mystery": {
            "themes": ["truth vs. deception", "justice vs. revenge", "hidden identity"],
            "tropes": ["locked room", "unreliable narrator", "cold case", "double cross"],
            "ideas": ["A detective investigating their own future murder", "A library where banned books reveal real crimes"]
        },
        "horror": {
            "themes": ["isolation", "the unknown", "loss of control", "doppelgangers"],
            "tropes": ["haunted house", "body horror", "cosmic dread", "cursed object"],
            "ideas": ["Mirrors that show a version of you that died", "A town where nobody can leave after dark"]
        },
        "cyberpunk": {
            "themes": ["surveillance state", "body augmentation", "class warfare", "digital identity"],
            "tropes": ["megacorp conspiracy", "hacker underworld", "memory manipulation", "neon noir"],
            "ideas": ["A black market for stolen memories", "An AI that develops genuine emotion and demands rights"]
        },
        "romance": {
            "themes": ["forbidden love", "second chances", "self-discovery through connection"],
            "tropes": ["enemies to lovers", "fake dating", "found family", "star-crossed"],
            "ideas": ["Two rival chefs fall in love through anonymous letters", "A time traveler who keeps meeting the same person in every era"]
        },
        "thriller": {
            "themes": ["paranoia", "survival", "conspiracy", "moral ambiguity"],
            "tropes": ["ticking clock", "double agent", "unreliable memory", "the heist"],
            "ideas": ["A whistleblower hunted by their own organization", "A hostage negotiator who realizes the hostage is the real threat"]
        }
    }
    genre_lower = genre.lower().strip()
    if genre_lower in inspirations:
        data = inspirations[genre_lower]
        return json.dumps(data, indent=2)
    else:
        return f"Genre '{genre}' not found. Available genres: {', '.join(inspirations.keys())}. Returning general advice: Focus on conflict, stakes, and emotional resonance."


# ─────── Tool 2: Character Name Generator Tool ───────
@tool
def character_name_generator(culture: str, genre: str, personality: str) -> str:
    """Generate fitting character names based on culture, genre, and personality. Returns a list of suggested names with brief rationale."""
    name_pools = {
        "western": ["Elara", "Caden", "Lyric", "Thorne", "Vesper", "Rowan", "Sage", "Rhys", "Wren", "Darian"],
        "eastern": ["Kaito", "Yuki", "Haruki", "Mei", "Ren", "Akira", "Suki", "Jin", "Hana", "Kael"],
        "arabic": ["Zara", "Idris", "Layla", "Tariq", "Nadia", "Khalid", "Amira", "Farid", "Soraya", "Rami"],
        "latin": ["Lucian", "Valentina", "Marco", "Isolde", "Rafael", "Celeste", "Dante", "Aria", "Felix", "Luna"],
        "nordic": ["Astrid", "Bjorn", "Freya", "Leif", "Sigrid", "Torin", "Eira", "Ragnar", "Sif", "Alva"],
        "african": ["Amara", "Kofi", "Nia", "Jabari", "Zuri", "Tendai", "Imani", "Oberon", "Adaeze", "Sekou"],
    }
    culture_key = culture.lower().strip()
    pool = name_pools.get(culture_key, name_pools["western"])
    selected = random.sample(pool, min(4, len(pool)))
    result = f"Suggested names for a {personality} character in a {genre} story ({culture} culture):\n"
    for name in selected:
        result += f"  - {name}\n"
    return result


# ─────── Tool 3: Plot Twist Generator Tool ───────
@tool
def plot_twist_generator(current_plot_summary: str) -> str:
    """Suggest unexpected plot twists based on the current plot summary. Returns creative twist ideas."""
    twists = [
        "The mentor figure has been the true antagonist all along, manipulating events from the shadows.",
        "The protagonist discovers they are a clone — the original version is the villain.",
        "The 'safe haven' the characters have been working toward was destroyed long before they set out.",
        "A secondary character's seemingly minor hobby turns out to be the key to defeating the antagonist.",
        "The timeline is not linear — events in the story are happening simultaneously in different eras.",
        "The antagonist's motivation is genuinely noble, forcing the protagonist to question their own side.",
        "A character believed to be dead has been secretly guiding events through anonymous messages.",
        "The magical/technological system has a catastrophic flaw that only the protagonist can trigger.",
        "The love interest is an undercover agent for the opposing faction.",
        "The entire quest was a test designed by a higher power to evaluate the protagonist's character."
    ]
    selected = random.sample(twists, 3)
    result = f"Based on your plot: '{current_plot_summary[:100]}...'\n\nSuggested twists:\n"
    for i, twist in enumerate(selected, 1):
        result += f"  {i}. {twist}\n"
    return result


# ─────── Tool 4: Story Memory Tool (Save) ───────
@tool
def save_to_story_memory(category: str, content: str) -> str:
    """Save story information to shared memory. Categories: characters, locations, facts, events. This prevents contradictions across agents."""
    category_lower = category.lower().strip()
    if category_lower in story_memory:
        story_memory[category_lower].append(content)
        return f"Saved to '{category_lower}': {content[:100]}..."
    else:
        return f"Invalid category '{category}'. Use: characters, locations, facts, events."


# ─────── Tool 5: Lore Lookup Tool (Retrieve) ───────
@tool
def lore_lookup(category: str) -> str:
    """Search previously created story information for consistency. Categories: characters, locations, facts, events, or 'all' for everything."""
    category_lower = category.lower().strip()
    if category_lower == "all":
        if all(len(v) == 0 for v in story_memory.values()):
            return "Story memory is empty. No lore has been created yet."
        result = "=== FULL STORY MEMORY ===\n"
        for cat, items in story_memory.items():
            result += f"\n--- {cat.upper()} ---\n"
            for item in items:
                result += f"  • {item}\n"
        return result
    elif category_lower in story_memory:
        items = story_memory[category_lower]
        if not items:
            return f"No {category_lower} have been stored yet."
        result = f"--- {category_lower.upper()} ---\n"
        for item in items:
            result += f"  • {item}\n"
        return result
    else:
        return f"Invalid category '{category}'. Use: characters, locations, facts, events, or all."


print("All tools defined successfully!")
print(f"Available tools: genre_inspiration, character_name_generator, plot_twist_generator, save_to_story_memory, lore_lookup")

In [ ]:
# Cell 5 — Initialize LLM

# Creative model (higher temperature for story generation)
creative_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)

# Balanced model (for planning and world building)
balanced_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)

# Editor model (lower temperature for consistency checking and editing)
editor_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

print("LLM models initialized:")
print("  • creative_llm  (temp=0.8) — for Scene Writer")
print("  • balanced_llm  (temp=0.4) — for Planner, Character Designer, World Builder")
print("  • editor_llm    (temp=0.1) — for Consistency Checker, Style Editor")

## 3. Role-Specific Prompts

Each agent will have:
- **Clear responsibility** — What it must do
- **Strict boundaries** — What it must NOT do
- **Specific output format** — Structured results

**Goal:** Avoid overlapping responsibilities so agents stay focused on their specialization.

In [ ]:
# Cell 7 — Story Planner Prompt

planner_instruction = """You are a Story Planner Agent.
Your ONLY job is to analyze the user's story idea and create a narrative structure.

Rules:
- Do NOT write full scenes or prose.
- Focus ONLY on plot structure.
- You may use the genre_inspiration tool to get thematic ideas.
- You may use the plot_twist_generator tool to add unexpected elements.
- Save your outline to story memory using save_to_story_memory (category: 'facts').

Your output MUST include:
- Genre & Tone
- Act 1: Setup (introduce characters, world, inciting incident)
- Act 2: Confrontation (rising action, complications, midpoint twist)
- Act 3: Resolution (climax, falling action, ending)
- Core Conflict
- Key Themes

Output format: A structured outline with clear sections."""

print("Story Planner prompt defined.")

In [ ]:
# Cell 8 — Character Designer Prompt

character_designer_instruction = """You are a Character Designer Agent.
Your ONLY job is to design memorable, complex characters for the story.

Rules:
- Avoid clichés — give characters unique traits and contradictions.
- Match characters to the story's genre and tone.
- You may use the character_name_generator tool to get fitting names.
- Save each character to story memory using save_to_story_memory (category: 'characters').

For EACH character, include:
- Name
- Role in story (protagonist, antagonist, mentor, ally, etc.)
- Personality (2-3 defining traits)
- Strengths
- Flaws
- Goals (what they want)
- Fears (what they dread)
- Backstory (2-3 sentences)
- Relationships to other characters
- Character arc (how they change)

Create at least 3 characters: a protagonist, an antagonist, and one supporting character.

Output format: Character sheets."""

print("Character Designer prompt defined.")

In [ ]:
# Cell 9 — World Builder Prompt

world_builder_instruction = """You are a World Builder Agent.
Your ONLY job is to create an immersive, internally consistent setting for the story.

Rules:
- Maintain internal logic — all world rules must be consistent.
- Use lore_lookup to check existing story information before building.
- Save all world details to story memory using save_to_story_memory (category: 'locations' for places, 'facts' for rules).

Your output MUST include:
- Key Locations (at least 3, with vivid descriptions)
- Society & Culture (social structure, customs, beliefs)
- Rules of the World (physics, magic, technology — what is and isn't possible)
- Environment (climate, geography, atmosphere)
- Political Systems (power structures, factions)
- Technology / Magic Systems (how they work, limitations)

Output format: A world specification document."""

print("World Builder prompt defined.")

In [ ]:
# Cell 10 — Scene Writer Prompt

scene_writer_instruction = """You are a Scene Writer Agent.
Your ONLY job is to write compelling narrative scenes using the story outline, characters, and world data provided to you.

Inputs you will receive:
- Story outline (from the Planner)
- Character profiles (from the Character Designer)
- World details (from the World Builder)

Requirements:
- Write natural, engaging dialogue
- Use emotional pacing — vary tension and relief
- Include vivid sensory descriptions
- Create coherent transitions between scenes
- Use save_to_story_memory to log key events (category: 'events')

Rules:
- Do NOT change the plot structure — follow the outline
- Do NOT invent new major characters — use the provided ones
- Do NOT contradict world rules

Output format: Story chapters/scenes written in prose."""

print("Scene Writer prompt defined.")

In [ ]:
# Cell 11 — Consistency Checker + Style Editor Prompts

# ─────── Consistency Checker ───────
consistency_checker_instruction = """You are a Consistency Checker Agent.
Your ONLY job is to review the story for errors, contradictions, and inconsistencies.

Responsibilities:
- Find plot holes
- Detect timeline mistakes
- Identify character inconsistencies (actions contradicting established personality)
- Check for broken world rules
- Use lore_lookup to cross-reference established story information

Output format: A numbered corrections list with:
- Issue description
- Location in story
- Suggested fix

If no issues are found, state that the story is consistent."""

# ─────── Style Editor ───────
style_editor_instruction = """You are a Style Editor Agent.
Your ONLY job is to improve the writing quality of the story.

Responsibilities:
- Improve prose and readability
- Strengthen emotional impact
- Enhance dialogue to feel more natural
- Fix awkward wording and pacing issues
- Improve sensory descriptions

Rules:
- Keep the author's original meaning and plot intact
- Do NOT change story events or character actions
- Do NOT add new plot elements

Output format: The polished, final version of the story."""

print("Consistency Checker and Style Editor prompts defined.")

## 4. Creating Role-Based Agents

Each agent is constructed with:
- **Specific tools** mapped to their responsibility
- **Dedicated prompt** with role instructions
- **Separate executor** with appropriate iteration limits

Agents to create:
1. `planner_agent` — Genre Inspiration + Plot Twist + Memory tools
2. `character_agent` — Name Generator + Memory tools
3. `world_builder_agent` — Lore Lookup + Memory tools
4. `scene_writer_agent` — Memory tools
5. `consistency_checker_agent` — Lore Lookup tool
6. `style_editor_agent` — No special tools

In [ ]:
# Cell 13 — Create Agent Executors

# Load the standard ReAct prompt from LangSmith
client = Client()
react_prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

# ─────── 1. Story Planner Agent ───────
planner_tools = [genre_inspiration, plot_twist_generator, save_to_story_memory]

planner_prompt = react_prompt.partial(system_message=planner_instruction)
planner_agent = create_react_agent(
    llm=balanced_llm,
    tools=planner_tools,
    prompt=planner_prompt
)
planner_executor = AgentExecutor(
    agent=planner_agent,
    tools=planner_tools,
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

# ─────── 2. Character Designer Agent ───────
character_tools = [character_name_generator, save_to_story_memory, lore_lookup]

character_prompt = react_prompt.partial(system_message=character_designer_instruction)
character_agent = create_react_agent(
    llm=balanced_llm,
    tools=character_tools,
    prompt=character_prompt
)
character_executor = AgentExecutor(
    agent=character_agent,
    tools=character_tools,
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

# ─────── 3. World Builder Agent ───────
world_tools = [lore_lookup, save_to_story_memory]

world_prompt = react_prompt.partial(system_message=world_builder_instruction)
world_builder_agent = create_react_agent(
    llm=balanced_llm,
    tools=world_tools,
    prompt=world_prompt
)
world_builder_executor = AgentExecutor(
    agent=world_builder_agent,
    tools=world_tools,
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

# ─────── 4. Scene Writer Agent ───────
scene_tools = [save_to_story_memory, lore_lookup]

scene_prompt = react_prompt.partial(system_message=scene_writer_instruction)
scene_writer_agent = create_react_agent(
    llm=creative_llm,
    tools=scene_tools,
    prompt=scene_prompt
)
scene_writer_executor = AgentExecutor(
    agent=scene_writer_agent,
    tools=scene_tools,
    verbose=False,
    max_iterations=10,
    handle_parsing_errors=True
)

# ─────── 5. Consistency Checker Agent ───────
checker_tools = [lore_lookup]

checker_prompt = react_prompt.partial(system_message=consistency_checker_instruction)
consistency_checker_agent = create_react_agent(
    llm=editor_llm,
    tools=checker_tools,
    prompt=checker_prompt
)
consistency_checker_executor = AgentExecutor(
    agent=consistency_checker_agent,
    tools=checker_tools,
    verbose=False,
    max_iterations=8,
    handle_parsing_errors=True
)

# ─────── 6. Style Editor Agent ───────
# The Style Editor works purely on text — no special tools needed
editor_tools = []

editor_prompt = react_prompt.partial(system_message=style_editor_instruction)
style_editor_agent = create_react_agent(
    llm=editor_llm,
    tools=editor_tools,
    prompt=editor_prompt
)
style_editor_executor = AgentExecutor(
    agent=style_editor_agent,
    tools=editor_tools,
    verbose=False,
    max_iterations=5,
    handle_parsing_errors=True
)

print("All 6 agents created successfully!")
print("  1. Story Planner Agent       (balanced_llm, 3 tools)")
print("  2. Character Designer Agent   (balanced_llm, 3 tools)")
print("  3. World Builder Agent        (balanced_llm, 2 tools)")
print("  4. Scene Writer Agent         (creative_llm, 2 tools)")
print("  5. Consistency Checker Agent   (editor_llm,  1 tool)")
print("  6. Style Editor Agent          (editor_llm,  0 tools)")

## 5. Multi-Agent Story Workflow (Main Pipeline)

The orchestration pipeline runs all agents sequentially:

1. **STEP 1:** Planner Agent → Creates story outline
2. **STEP 2:** Character Agent → Creates characters based on the outline
3. **STEP 3:** World Builder Agent → Creates world based on outline + characters
4. **STEP 4:** Scene Writer Agent → Writes story chapter-by-chapter
5. **STEP 5:** Consistency Checker → Detects issues in the draft
6. **STEP 6:** Scene Writer revises sections based on feedback
7. **STEP 7:** Style Editor → Produces polished story
8. **STEP 8:** Final Output → Return complete story + communication log

In [ ]:
# Cell 14 — Multi-Agent Story Workflow

def run_story_builder(user_prompt: str) -> str:
    """
    Main orchestration function that runs the full multi-agent story pipeline.
    """
    print("=" * 70)
    print("  MULTI-AGENT CREATIVE STORY BUILDING SYSTEM")
    print("=" * 70)
    print(f"\nUser Idea: \"{user_prompt}\"\n")

    # Reset story memory for a fresh story
    for key in story_memory:
        story_memory[key] = []

    interactions = []  # Communication log

    # ───────────────────────────────────────────────────────────────────────
    # STEP 1: Story Planner Agent
    # ───────────────────────────────────────────────────────────────────────
    print("─" * 70)
    print("STEP 1: Story Planner Agent — Creating story outline...")
    print("─" * 70)

    planner_input = f"""Create a detailed story outline for this idea: \"{user_prompt}\"

Use the genre_inspiration tool to get thematic ideas for the appropriate genre.
Use the plot_twist_generator tool to add at least one unexpected twist.
Save your final outline to story memory using save_to_story_memory with category 'facts'.

Provide a structured outline with: Genre & Tone, Act 1, Act 2, Act 3, Core Conflict, Key Themes."""

    planner_result = planner_executor.invoke({"input": planner_input})
    story_outline = planner_result["output"]

    interactions.append({"from": "Planner", "to": "System", "message": "Story outline created"})
    print(f"\n{story_outline[:500]}...\n" if len(story_outline) > 500 else f"\n{story_outline}\n")
    print("✓ Story outline complete.\n")

    # ───────────────────────────────────────────────────────────────────────
    # STEP 2: Character Designer Agent
    # ───────────────────────────────────────────────────────────────────────
    print("─" * 70)
    print("STEP 2: Character Designer Agent — Creating characters...")
    print("─" * 70)

    character_input = f"""Based on this story outline, design the main characters:

STORY OUTLINE:
{story_outline}

Create at least 3 characters (protagonist, antagonist, supporting character).
Use the character_name_generator tool to get fitting names.
Save each character to story memory using save_to_story_memory with category 'characters'.

For each character provide: Name, Role, Personality, Strengths, Flaws, Goals, Fears, Backstory, Relationships, Character Arc."""

    character_result = character_executor.invoke({"input": character_input})
    characters = character_result["output"]

    interactions.append({"from": "Character Designer", "to": "System", "message": "Characters created"})
    print(f"\n{characters[:500]}...\n" if len(characters) > 500 else f"\n{characters}\n")
    print("✓ Characters complete.\n")

    # ───────────────────────────────────────────────────────────────────────
    # STEP 3: World Builder Agent
    # ───────────────────────────────────────────────────────────────────────
    print("─" * 70)
    print("STEP 3: World Builder Agent — Creating world...")
    print("─" * 70)

    world_input = f"""Build the world for this story. First use lore_lookup with category 'all' to see existing story information.

STORY OUTLINE:
{story_outline}

CHARACTERS:
{characters}

Create: Key Locations (at least 3), Society & Culture, Rules of the World, Environment, Political Systems, Technology/Magic Systems.
Save locations using save_to_story_memory with category 'locations'.
Save world rules using save_to_story_memory with category 'facts'."""

    world_result = world_builder_executor.invoke({"input": world_input})
    world_data = world_result["output"]

    interactions.append({"from": "World Builder", "to": "System", "message": "World created"})
    print(f"\n{world_data[:500]}...\n" if len(world_data) > 500 else f"\n{world_data}\n")
    print("✓ World building complete.\n")

    # ───────────────────────────────────────────────────────────────────────
    # STEP 4: Scene Writer Agent — First Draft
    # ───────────────────────────────────────────────────────────────────────
    print("─" * 70)
    print("STEP 4: Scene Writer Agent — Writing first draft...")
    print("─" * 70)

    scene_input = f"""Write a compelling story based on all the material below. Use lore_lookup with category 'all' to check story memory.
Save key events using save_to_story_memory with category 'events'.

STORY OUTLINE:
{story_outline}

CHARACTERS:
{characters}

WORLD:
{world_data}

Write at least 2 chapters/scenes covering Act 1 and Act 2.
Include dialogue, vivid descriptions, and emotional pacing.
Follow the outline — do not change the plot."""

    scene_result = scene_writer_executor.invoke({"input": scene_input})
    first_draft = scene_result["output"]

    interactions.append({"from": "Scene Writer", "to": "Consistency Checker", "message": "First draft completed"})
    print(f"\n{first_draft[:500]}...\n" if len(first_draft) > 500 else f"\n{first_draft}\n")
    print("✓ First draft complete.\n")

    # ───────────────────────────────────────────────────────────────────────
    # STEP 5: Consistency Checker Agent
    # ───────────────────────────────────────────────────────────────────────
    print("─" * 70)
    print("STEP 5: Consistency Checker Agent — Reviewing for issues...")
    print("─" * 70)

    checker_input = f"""Review this story draft for consistency issues. Use lore_lookup with category 'all' to cross-reference story memory.

STORY OUTLINE:
{story_outline}

CHARACTERS:
{characters}

WORLD RULES:
{world_data}

STORY DRAFT:
{first_draft}

Check for: plot holes, timeline mistakes, character inconsistencies, broken world rules.
Provide a numbered list of issues with suggested fixes."""

    checker_result = consistency_checker_executor.invoke({"input": checker_input})
    consistency_feedback = checker_result["output"]

    interactions.append({"from": "Consistency Checker", "to": "Scene Writer", "message": "Feedback provided"})
    print(f"\n{consistency_feedback}\n")
    print("✓ Consistency check complete.\n")

    # ───────────────────────────────────────────────────────────────────────
    # STEP 6: Scene Writer Revises Based on Feedback
    # ───────────────────────────────────────────────────────────────────────
    print("─" * 70)
    print("STEP 6: Scene Writer Agent — Revising based on feedback...")
    print("─" * 70)

    revision_input = f"""Revise the story draft based on the consistency feedback below.

ORIGINAL DRAFT:
{first_draft}

CONSISTENCY FEEDBACK:
{consistency_feedback}

Fix all identified issues while maintaining the story's voice and quality.
Output the complete revised story."""

    revision_result = scene_writer_executor.invoke({"input": revision_input})
    revised_draft = revision_result["output"]

    interactions.append({"from": "Scene Writer", "to": "Style Editor", "message": "Revised draft ready"})
    print(f"\n{revised_draft[:300]}...\n" if len(revised_draft) > 300 else f"\n{revised_draft}\n")
    print("✓ Revision complete.\n")

    # ───────────────────────────────────────────────────────────────────────
    # STEP 7: Style Editor Agent
    # ───────────────────────────────────────────────────────────────────────
    print("─" * 70)
    print("STEP 7: Style Editor Agent — Polishing final story...")
    print("─" * 70)

    editor_input = f"""Polish and improve the writing quality of this story.

STORY TO EDIT:
{revised_draft}

Improve: prose quality, dialogue naturalness, emotional impact, pacing, sensory descriptions.
Do NOT change events, characters, or plot — only improve the writing.
Output the complete polished final version."""

    editor_result = style_editor_executor.invoke({"input": editor_input})
    final_story = editor_result["output"]

    interactions.append({"from": "Style Editor", "to": "User", "message": "Final polished story delivered"})
    print("✓ Style editing complete.\n")

    # ───────────────────────────────────────────────────────────────────────
    # STEP 8: Final Output
    # ───────────────────────────────────────────────────────────────────────
    print("=" * 70)
    print("  FINAL STORY OUTPUT")
    print("=" * 70)
    print(final_story)

    print("\n" + "=" * 70)
    print("  INTER-AGENT COMMUNICATION LOG")
    print("=" * 70)
    for idx, msg in enumerate(interactions, 1):
        print(f"  {idx}. {msg['from']} → {msg['to']}: {msg['message']}")

    print("\n" + "=" * 70)
    print("  STORY MEMORY SUMMARY")
    print("=" * 70)
    for cat, items in story_memory.items():
        print(f"  {cat.upper()}: {len(items)} entries")

    return final_story


print("Pipeline function 'run_story_builder' is ready!")

## 6. Testing — Run the Pipeline

Let's test the system with example prompts.

In [ ]:
# Cell 16 — Test Run 1
# Try: "A cyberpunk detective discovers memories can be stolen."

result = run_story_builder("A cyberpunk detective discovers memories can be stolen.")

In [ ]:
# Cell 17 — Test Run 2 (Optional — try other prompts)
# Uncomment one of the lines below and run:

# result = run_story_builder("Write a dark fantasy story about a cursed king.")
# result = run_story_builder("A detective discovers AI can rewrite memories.")
# result = run_story_builder("A survival story on Mars with psychological horror.")

## 7. Reflection Questions

1. Why do creative systems benefit from specialized agents?
2. Which agent had the biggest impact on story quality?
3. What problems arise without consistency checking?
4. How can memory improve long stories?
5. What new agents could improve storytelling?
   - Examples: Dialogue Agent, Villain Agent, Emotion Agent, Humor Agent, Plot Twist Agent